# 🔍 تحليل Sprint 8 — XAI (الخطوة 1 + 2)

---

## 🎯 TL;DR

> **XAI** يجعل النموذج **قابلاً للتفسير**: أي مؤشر دفع القرار؟ بأي قوة؟ (`feature × weight`). **`get_feature_weights`** يستخرج `coef_` من LogisticRegression. **`PredictionExplainer`** يحلل: Bullish vs Bearish + Global Importance.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[LogisticForecaster] --> B[model.coef_]
    B --> C[get_feature_weights]
    C --> D[list float]
    E[PredictionExplainer.explain] --> F[features]
    E --> G[weights]
    F --> H[zip with FEATURE_NAMES]
    G --> H
    H --> I[contribution = val × weight]
    I --> J{Bullish vs Bearish}
    J --> K[top_bullish_factors]
    J --> L[top_bearish_factors]
    J --> M[global_importance = abs weight]

    style E fill:#bfb,stroke:#333,stroke-width:3px
    style I fill:#bdf,stroke:#333,stroke-width:2px
    style K fill:#bbf,stroke:#333
    style L fill:#fdd,stroke:#333
```

---

## 🔬 التشريح

### 1️⃣ `get_feature_weights` — استخراج المعاملات

```python
if not self.is_trained:
    raise ValidationError("لا يمكن قراءة أوزان النموذج قبل التدريب.")
return [float(c) for c in self.model.coef_[0]]
```

| العنصر | الدور |
|--------|-------|
| `is_trained` | Fail Early — لا أوزان قبل التدريب |
| `coef_` | مصفوفة `(1, n_features)` في sklearn |
| `[0]` | الصف الأول (نموذج binary) |
| `float(c)` | تحويل من `np.float64` لـ `float` |

**🎯 `coef_` — معاملات Logistic:**

| القيمة | المعنى |
|--------|--------|
| **موجب** | يرجّح الصعود (`P(Up)` يزيد) |
| **سالب** | يرجّح الهبوط (`P(Up)` يقل) |
| **صفر** | لا تأثير (نادراً) |
| **القيمة المطلقة** | قوة التأثير |

**مثال:** `w = [1.5, -0.8, -0.2, 0.9, 1.2]` → `Price Return` أقوى مؤيد للصعود، `SMA Distance` أقوى معارض.

---

### 2️⃣ `ml/explainability.py` — محرك التفسير

#### `FEATURE_NAMES` — الأسماء المعروضة

```python
FEATURE_NAMES = [
    "Price Return",       # ← 1: price_return
    "SMA Distance",       # ← 2: sma_diff
    "Candle Range",       # ← 3: candle_range
    "RSI (Normalized)",   # ← 4: rsi_norm
    "MACD",               # ← 5: macd_norm
]
```

**🎯 تريكة:** `FeatureExtractor` ينتج **5 features بدون أسماء** (float array). `FEATURE_NAMES` يربطها بأسماء بشرية — **ترتيب حتمي** (مطابق لـ `extract_features`).

**⚠️ خطر:** لو `FeatureExtractor` أضاف/غيّر ترتيب feature → `FEATURE_NAMES` يجب أن يتزامن. **الحل الأفضل:** ربطه بـ `FeatureExtractor.FEATURE_NAMES` (class constant).

---

#### `FeatureImpact` — تأثير ميزة واحدة

```python
@dataclass
class FeatureImpact:
    name: str
    feature_value: float
    weight: float
    contribution: float          # = feature_value × weight
    impact_type: str             # "BULLISH" | "BEARISH"
```

**🎯 `contribution = val × w`:** هذا هو **التأثير الموضعي (Local)** — كم ساهمت هذه الميزة في هذا القرار بالذات.

**مثال:**
| الميزة | Value | Weight | Contribution |
|--------|-------|--------|--------------|
| Price Return | 0.003 | +1.5 | +0.0045 |
| SMA Distance | 0.015 | −0.8 | **−0.0120** |
| RSI | 0.65 | +0.9 | +0.5850 |

**التفسير:** RSI هو **أكبر دافع للصعود** (+0.585)، بينما SMA Distance **أكبر معارض** (−0.012).

---

#### `ExplanationResult` — التقرير

| الحقل | المحتوى |
|-------|---------|
| `top_bullish_factors` | الميزات ذات `contribution > 0`، مرتبة تنازلياً |
| `top_bearish_factors` | الميزات ذات `contribution < 0`، مرتبة بـ `abs` تنازلياً |
| `global_importance` | `{name: abs(weight)}` — **الأهمية الشاملة** |

**🎯 `to_dict()`:** يحتفظ فقط بـ `name` + `contribution` (round 4) — نظيف للـ JSON/CLI.

---

#### `explain()` — قلب المحرك

```mermaid
graph TD
    A[explain features weights] --> B{len match?}
    B -->|No| C[ValueError]
    B -->|Yes| D[zip names, values, weights]
    D --> E[contrib = val × weight]
    E --> F[impact_type = BULLISH/BEARISH]
    F --> G[append FeatureImpact]
    G --> H[global_imp name = abs weight]
    H --> D
    D --> I[split bullish / bearish]
    I --> J[sort bullish DESC by contrib]
    I --> K[sort bearish DESC by abs contrib]
    J --> L[ExplanationResult]
    K --> L

    style E fill:#bdf,stroke:#333,stroke-width:2px
    style L fill:#bfb,stroke:#333,stroke-width:2px
```

**الخطوات:**

| # | الخطوة | السبب |
|---|--------|-------|
| 1 | `len(features) == len(weights) == 5` | Fail Early |
| 2 | `zip(NAMES, features, weights)` | ربط الأسماء |
| 3 | `contrib = val × w` | المساهمة الموضعية |
| 4 | `impact_type` | تصنيف |
| 5 | `global_imp[name] = abs(w)` | الأهمية الشاملة |
| 6 | `split + sort` | Top factors |

**🎯 `@staticmethod`:** `explain` لا تحتاج `self` — **stateless**، يمكن استدعاؤها بدون إنشاء كائن.

---

## 🧠 Global vs Local — **مفهوم XAI الأساسي**

| النوع | السؤال | من أين؟ | مثال |
|-------|--------|---------|------|
| **Global** | أي الميزات **عموماً** أهم؟ | `abs(weight)` | "RSI له وزن 0.9 → مهم دائماً" |
| **Local** | أي الميزات **الآن** دفعت القرار؟ | `value × weight` | "RSI = 0.65 × 0.9 = +0.585 → دفع صعوداً **الآن**" |

**الفرق الحرج:**

| الحالة | Global | Local |
|--------|--------|-------|
| RSI weight=0.9، value=0.50 (محايد) | 0.9 (**مهم**) | 0.45 (**مساهمة ضعيفة**) |
| RSI weight=0.9، value=0.95 (ذروة) | 0.9 (**مهم**) | 0.855 (**مساهمة قوية**) |

**القاعدة:** Global = **قدرة التأثير**. Local = **التأثير الفعلي في هذه اللحظة**.

---

## 🧪 التحقق (وصف)

**المدخلات:**
```
features = [0.003, 0.015, 0.004, 0.65, 0.002]
weights  = [1.5,  -0.8,  -0.2,  0.9,  1.2]
```

**الحساب:**

| Feature | Val | W | Contrib | Type |
|---------|-----|---|---------|------|
| Price Return | 0.003 | +1.5 | **+0.0045** | BULLISH |
| SMA Distance | 0.015 | −0.8 | **−0.0120** | BEARISH |
| Candle Range | 0.004 | −0.2 | **−0.0008** | BEARISH |
| RSI | 0.65 | +0.9 | **+0.5850** | BULLISH |
| MACD | 0.002 | +1.2 | **+0.0024** | BULLISH |

**Top Bullish:** RSI (0.585) > Price Return (0.0045) > MACD (0.0024)
**Top Bearish:** SMA Distance (−0.012) > Candle Range (−0.0008)
**Global:** MACD (1.2) > Price Return (1.5) ⚠️ — **تصحيح:** Price Return (1.5) > MACD (1.2) > RSI (0.9) > SMA (0.8) > Candle (0.2)

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| **`FEATURE_NAMES` مكرر** | يجب أن يطابق `FeatureExtractor` — **خطر sync** |
| `round(v, 4)` في `to_dict` | يفقد دقة MACD الصغيرة |
| لا `base_value` (intercept) | SHAP يضيفه، هنا **ناقص** — مجموع contributions ≠ logit كامل |
| `BULLISH if contrib >= 0` | صفر يُصنف BULLISH — **قرار تصميمي** |
| `@staticmethod` | لا يدعم نماذج مختلفة — **YAGNI** |
| `float(c)` في weights | يفقد دقة Decimal — **مقبول** لـ sklearn |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Transparency** | تفسير كل قرار |
| **Global vs Local** | أهمية شاملة + مساهمة موضعية |
| **Fail Early** | `is_trained` + `len match` |
| **Stateless** | `@staticmethod` |
| **Data Structures** | `FeatureImpact` + `ExplanationResult` |
| **JSON Ready** | `to_dict()` |
| **Interpretability** | `contribution = val × w` |

---

## 🗺️ Sprint 8

```mermaid
graph LR
    A["1. get_feature_weights ✅"] --> B["2. ml/explainability.py ✅"]
    B --> C["3. ForecastService + XAI"]
    C --> D["4. CLI: --explain flag"]
    D --> E["5. tests + docs"]

    style B fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 3**: دمج `PredictionExplainer` في `ForecastService` + `ForecastReport`.

**التغييرات:**
1. `ForecastReport` + `explanation: ExplanationResult | None = None`
2. `generate_forecast(..., explain: bool = False)`:
   - لو `explain`: `weights = model.get_feature_weights()` → `PredictionExplainer.explain(latest_features, weights)`
   - ضم `explanation` في التقرير
3. `to_dict` يُضم `explanation` (لو موجود).

**الخطوة 4**: `forex forecast --explain` → يمرر `explain=True`.

**Output متوقع للـ CLI:**

```
--- XAI Explanation ---
🟢 Top Bullish Drivers:
  1. RSI (Normalized)  : +0.5850
  2. Price Return      : +0.0045
  3. MACD              : +0.0024

🔴 Top Bearish Drags:
  1. SMA Distance      : -0.0120
  2. Candle Range      : -0.0008

Global Importance:
  Price Return: 1.50  ← الأكثر تأثيراً عموماً
  MACD:         1.20
  RSI:          0.90
  SMA Distance: 0.80
  Candle Range: 0.20
```

**السؤال الجوهري:** هل `contribution` مجموعها = 0 (neutral)؟

**الجواب:** **لا** — مجموعها يساوي `logit - intercept` (بدون bias). لتفسير كامل (SHAP-style)، نحتاج إضافة `base_value` (intercept). **Sprint 8** يكفي للـ Local explanation. **SHAP** الكامل في Sprint 9+ إن لزم.

In [1]:
from ml.explainability import PredictionExplainer

# محاكاة مدخلات لشمعة وأوزان نموذج
features = [0.003, 0.015, 0.004, 0.65, 0.002]
weights = [1.5, -0.8, -0.2, 0.9, 1.2]

explanation = PredictionExplainer.explain(features, weights)

print('--- Top Bullish Drivers ---')
for f in explanation.top_bullish_factors:
    print(f'  + {f.name}: contrib={f.contribution:.4f} (Weight={f.weight:.2f})')

print('\n--- Top Bearish Drags ---')
for f in explanation.top_bearish_factors:
    print(f'  - {f.name}: contrib={f.contribution:.4f} (Weight={f.weight:.2f})')

--- Top Bullish Drivers ---
  + RSI (Normalized): contrib=0.5850 (Weight=0.90)
  + Price Return: contrib=0.0045 (Weight=1.50)
  + MACD: contrib=0.0024 (Weight=1.20)

--- Top Bearish Drags ---
  - SMA Distance: contrib=-0.0120 (Weight=-0.80)
  - Candle Range: contrib=-0.0008 (Weight=-0.20)
